In [1]:
import os
import tensorflow as tf
tf.executing_eagerly() #check

True

In [2]:
class LinearModel(tf.keras.Model):
    def __init__(self, n_vals=2):
        super(LinearModel, self).__init__()
        self.denseout = tf.keras.layers.Dense(units=n_vals, input_shape=(1, ), activation='linear')

    def call(self, input):
        result = self.denseout(input)
        return result

In [3]:
# [features] -> label
fake_x = [[1.], [3], [5], [2], [8], [9], [10], [1.2], [4], [7], [-1.2], [0.01], [-38.12]] 
fake_y = list(map(lambda x: 1.2*x[0]+0.3, fake_x))

size_fake_data = len(fake_x)
fulltrain_fake_dataset = tf.data.Dataset.from_tensor_slices((fake_x, fake_y))
train_size = int(0.8 * size_fake_data)

train_fake_dataset = fulltrain_fake_dataset.take(train_size)
val_fake_dataset = fulltrain_fake_dataset.skip(train_size).batch(1)
fulltrain_fake_dataset = fulltrain_fake_dataset.batch(1)

train_fake_dataset = train_fake_dataset.shuffle(100, seed=123).batch(2)

In [5]:
model = LinearModel(1)
optimizer = tf.keras.optimizers.Adagrad(learning_rate=100)
#optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
train_loss = tf.keras.metrics.Mean(name='train_loss')
valid_loss = tf.keras.metrics.Mean(name='valid_loss')

@tf.function
def train_step(features, labels):
  with tf.GradientTape() as tape:
    predictions = tf.squeeze(model(features), axis=1)
    loss = tf.keras.losses.mean_squared_error(labels, predictions)
  gradients = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(gradients, model.trainable_variables))

  train_loss(loss)

@tf.function
def valid_step(features, labels):
  with tf.GradientTape() as tape:
    predictions = tf.squeeze(model(features), axis=1)
    loss = tf.keras.losses.mean_squared_error(labels, predictions)

  valid_loss(loss)


EPOCHS = 40
epochs_pervalid = 5

for epoch in range(EPOCHS):
  train_loss.reset_states()
  valid_loss.reset_states()

  for features, labels in train_fake_dataset:
    train_step(features, labels)

  if epoch % epochs_pervalid == epochs_pervalid - 1:
    for features, labels in train_fake_dataset:
      valid_step(features, labels)
    template = 'Epoch {}, Loss: {}, Valid Loss: {}'
    print(template.format(epoch + 1,
                          train_loss.result(),
                          valid_loss.result()))

Epoch 5, Loss: 4.951594352722168, Valid Loss: 1.2202606201171875
Epoch 10, Loss: 2.998828411102295, Valid Loss: 18.553848266601562
Epoch 15, Loss: 266.3484191894531, Valid Loss: 398.8363952636719
Epoch 20, Loss: 4918.0224609375, Valid Loss: 168.93099975585938
Epoch 25, Loss: 6.782103061676025, Valid Loss: 40.59613037109375
Epoch 30, Loss: 1.3736035823822021, Valid Loss: 0.4126364290714264
Epoch 35, Loss: 0.15761800110340118, Valid Loss: 0.07667559385299683
Epoch 40, Loss: 0.0002093117218464613, Valid Loss: 2.4392134946538135e-05


In [12]:
#fake_y = list(map(lambda x: 1.2*x[0]+0.3, fake_x))
model.get_weights()

[array([[1.1986432]], dtype=float32), array([0.30430725], dtype=float32)]

In [14]:
for features, labels in fulltrain_fake_dataset:
  with tf.GradientTape() as tape:
    predictions = tf.squeeze(model(features), axis=1)
    loss = tf.keras.losses.mean_squared_error(labels, predictions)
    print("predictions = ", predictions)
    print("labels = ", labels)
    print("loss = ", loss)
    print()

predictions =  tf.Tensor([1.5029504], shape=(1,), dtype=float32)
labels =  tf.Tensor([1.5], shape=(1,), dtype=float32)
loss =  tf.Tensor(8.705037e-06, shape=(), dtype=float32)

predictions =  tf.Tensor([3.9002368], shape=(1,), dtype=float32)
labels =  tf.Tensor([3.9], shape=(1,), dtype=float32)
loss =  tf.Tensor(5.6050396e-08, shape=(), dtype=float32)

predictions =  tf.Tensor([6.2975235], shape=(1,), dtype=float32)
labels =  tf.Tensor([6.3], shape=(1,), dtype=float32)
loss =  tf.Tensor(6.1340043e-06, shape=(), dtype=float32)

predictions =  tf.Tensor([2.7015936], shape=(1,), dtype=float32)
labels =  tf.Tensor([2.7], shape=(1,), dtype=float32)
loss =  tf.Tensor(2.5395284e-06, shape=(), dtype=float32)

predictions =  tf.Tensor([9.893453], shape=(1,), dtype=float32)
labels =  tf.Tensor([9.9], shape=(1,), dtype=float32)
loss =  tf.Tensor(4.286287e-05, shape=(), dtype=float32)

predictions =  tf.Tensor([11.092095], shape=(1,), dtype=float32)
labels =  tf.Tensor([11.1], shape=(1,), dtype=fl

In [6]:
for features, labels in val_fake_dataset:
  with tf.GradientTape() as tape:
    predictions = tf.squeeze(model(features), axis=1)
    loss = tf.keras.losses.mean_squared_error(labels, predictions)
    print("predictions = ", predictions)
    print("labels = ", labels)
    print("loss = ", loss)

predictions =  tf.Tensor([-1.1340647], shape=(1,), dtype=float32)
labels =  tf.Tensor([-1.14], shape=(1,), dtype=float32)
loss =  tf.Tensor(3.522792e-05, shape=(), dtype=float32)
predictions =  tf.Tensor([0.3162937], shape=(1,), dtype=float32)
labels =  tf.Tensor([0.312], shape=(1,), dtype=float32)
loss =  tf.Tensor(1.843569e-05, shape=(), dtype=float32)
predictions =  tf.Tensor([-45.38797], shape=(1,), dtype=float32)
labels =  tf.Tensor([-45.444], shape=(1,), dtype=float32)
loss =  tf.Tensor(0.0031393915, shape=(), dtype=float32)


In [8]:
for features, labels in train_fake_dataset:
  with tf.GradientTape() as tape:
    predictions = tf.squeeze(model(features), axis=1)
    loss = tf.keras.losses.mean_squared_error(labels, predictions)
    print("predictions = ", predictions)
    print("labels = ", labels)
    print("loss = ", loss)

predictions =  tf.Tensor([1.5029504 9.893453 ], shape=(2,), dtype=float32)
labels =  tf.Tensor([1.5 9.9], shape=(2,), dtype=float32)
loss =  tf.Tensor(2.5783953e-05, shape=(), dtype=float32)
predictions =  tf.Tensor([2.7015936 3.9002368], shape=(2,), dtype=float32)
labels =  tf.Tensor([2.7 3.9], shape=(2,), dtype=float32)
loss =  tf.Tensor(1.2977894e-06, shape=(), dtype=float32)
predictions =  tf.Tensor([8.69481   5.0988803], shape=(2,), dtype=float32)
labels =  tf.Tensor([8.7 5.1], shape=(2,), dtype=float32)
loss =  tf.Tensor(1.4094276e-05, shape=(), dtype=float32)
predictions =  tf.Tensor([12.290739   6.2975235], shape=(2,), dtype=float32)
labels =  tf.Tensor([12.3  6.3], shape=(2,), dtype=float32)
loss =  tf.Tensor(4.5951278e-05, shape=(), dtype=float32)
predictions =  tf.Tensor([11.092095   1.7426791], shape=(2,), dtype=float32)
labels =  tf.Tensor([11.1   1.74], shape=(2,), dtype=float32)
loss =  tf.Tensor(3.4833378e-05, shape=(), dtype=float32)


In [10]:
((1.5029504-1.5)**2 + (9.9-9.893453)**2)/2

2.5784034580007852e-05